In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion
import pandas as pd


In [11]:
songs = pd.read_pickle("./data/Songs")
vc = VocalAssistant(1)

In [10]:
device = 'cpu'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "model_checkpoint_sampled.pth"
custom_model, custom_processor = load_trained_model(custom_model_name)

/Users/lucabellani/Documents/UNI/Tesi/Recommersion/vocal_assistant/emotion/predict_emotion.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(chec

Loaded trained model from checkpoint.


In [15]:
vc.talk("What is your mood today?")
while True:
    command, vocal_file = vc.take_command()
    print(command)
    break

print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0]
print(audeering)
print("Custom: ")
custom = list(predict_emotion(custom_model, custom_processor, vocal_file).values())
print(custom)
#custom model seems to give the same results: overfitting?
# [0.4621863067150116, 0.5704010128974915, 0.615027904510498]


  listening....
Sample rate: 16000
Numpy array shape: (138577,)
i am happy
Audeering: 
[0.30133402 0.32972124 0.5233015 ]
Custom: 
[0.5339343547821045, 0.654996395111084, 0.6419190168380737]


In [13]:
import numpy as np
dim_vec = np.array(audeering[0:2])
songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

songs_list = songs_list.sort_values(by="eucl_dist")[:5]
songs_list

#TODO
#Careful that in the dataset there are some duplicates
#In a Deam Metadata file there titles with \t

,id,eucl_dist,Valence,Arousal,title,artist,mp3_file
320,1375,0.401481,0.100000,0.200000,\tBit a Bullet\t,Mr. & Mrs. Smith\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
323,1379,0.408629,0.100000,0.211111,\tChantiers Navals 412\t,LJ Kruzer\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
260,1310,0.412214,0.122222,0.188889,\tNeighborhood Funeral Dress\t,Mr. & Mrs. Smith\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
367,1444,0.418137,0.144444,0.166667,\tPeople Living And Leaving Big Citys\t,Augustus Bro & Gallery Six\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
509,1660,0.418884,0.122222,0.200000,\t05 CherryBlossom\t,Daddy_Scrabble\t,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [14]:
import sounddevice as sd

for i in range(len(songs_list)):
    sd.play(songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()


KeyboardInterrupt: 